# Spotify Genre Classification Analysis

This notebook contains the complete analysis pipeline for classifying Spotify tracks by genre using machine learning models.

The notebook is organized into the following sections:
1. **Exploratory Data Analysis (EDA)** - Data loading, cleaning, and feature exploration
2. **Framework Setup** - Common utilities and preprocessing pipeline
3. **Model Definitions** - Three different classification models
4. **Model Training and Evaluation** - Training and evaluating all models
5. **Results** - Confusion matrices and performance metrics

## Setup and Imports

In [ ]:
# Import all necessary libraries
import pandas as pd
import numpy as np
import seaborn as sn
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Scikit-learn imports
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn import preprocessing
from sklearn.preprocessing import TargetEncoder, StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.inspection import permutation_importance
from sklearn.metrics.pairwise import euclidean_distances

# SciPy for distributions
from scipy.stats import loguniform, randint, uniform

# 1. Exploratory Data Analysis (EDA)

In [ ]:
# Load the dataset
DATA_PATH = 'Spotify_tracks_final.csv'
df = pd.read_csv(DATA_PATH, index_col=0)

# Display columns
print("Dataset columns:")
print(df.columns.tolist())

In [ ]:
# Aantal dubbele tracknamen berekenen
duplicates = df['track_name'].duplicated().sum()
print(f'Duplicates: {duplicates}')

# Dubbele tracks verwijderen, alle kolommen behouden
df = df.drop_duplicates(subset=['track_name'])

In [ ]:
# Unieke genres weergeven
print(f"Unique genres:\n{df['genre'].unique()}\n")

### Feature Analysis

Voor de eerste analyse kijken we naar de numerieke features.

In [ ]:
numeric_features = df.select_dtypes(include=np.number).columns.tolist()
numeric_features

### Genre Similarity Analysis

In deze dataset zijn er enorm veel genres. Veel van deze genres lijken ook sterk op elkaar.
Een machine learning model zal dus redelijk wat problemen hebben bij het onderscheiden van bijvoorbeeld hard-rock en alt-rock.

Let's group rock genres and analyze their similarity based on audio features.

In [ ]:
ROCK_GENRES = [
    "alt-rock",
    "alternative",
    "hard-rock",
    "rock",
    "punk",
    "punk-rock",
    "rock-n-roll",
    "rockabilly",
    "groove",
    "grunge",
    "goth",
    "hardcore",
    "psych-rock",
    "power-pop",
    "indie-rock",
    "emo",
    "j-rock",
    "guitar"
]

df_rock = df[df['genre'].isin(ROCK_GENRES)]
genre_means = df_rock.groupby('genre')[numeric_features].mean()

genre_scaled = StandardScaler().fit_transform(genre_means)

distance_matrix = euclidean_distances(genre_scaled)
distance_df = pd.DataFrame(distance_matrix,
                           index=genre_means.index,
                           columns=genre_means.index)

In [ ]:
# Plot and save genre similarity heatmap
plt.figure(figsize=(12, 10))
sn.heatmap(distance_df, cmap='YlGnBu',
            xticklabels=True, yticklabels=True)
plt.title('Genre similarity')
plt.tight_layout()
plt.savefig('./eda_visualisations/genre_similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()

# Display the saved image
display(Image(filename='./eda_visualisations/genre_similarity_heatmap.png'))

**Observation:** Buiten het donkerblauwe kruis op "guitar" zijn meeste genres binnen de rockfamilie redelijk gelijkaardig. Tijdens het testen van vorige modellen struikelde het model ook het sterkst over genres die sterk bij elkaar liggen. Voor deze oefening zullen we vervolgens meerdere genres samen groeperen.

### Feature Correlation Analysis

Kolommen die verwijdert kunnen worden:
De eerste en meest duidelijke kolom die verwijdert kan worden zijn de verschillende tijdkolommen. Deze geven exact dezelfde waarden maar zijn gewoon een verschillende precisie.
Bijgevolg nemen we enkel duration_ms mee.

In [ ]:
# Heatmap of duration features correlation
plt.figure(figsize=(8, 6))
heatmap = sn.heatmap(df[['duration_seconds','duration_minutes','duration_ms']].corr(), 
                       annot=True, cmap="Blues", fmt='.1g', cbar=False)
plt.title('Duration Features Correlation')
plt.savefig(f"./eda_visualisations/heatmap_duration.png", dpi=150, bbox_inches="tight")
plt.close()

display(Image(filename='./eda_visualisations/heatmap_duration.png'))

Andere kolommen die weg kunnen gelaten worden zijn kolommen die randinformatie geven over een track. Dit gaat dan bijvoorbeeld over de track name, de artist, uit welk album het kwam.
Deze gegevens zijn moeilijk om te zetten naar numerieke waarden. Hoewel een artist of een album duiding kan geven over uit welk genre het kwam, zal het model wellicht geen voorspellingen kunnen doen op basis van deze gegevens als het bv. een nieuw nog niet gekend album of artist tegenkomt.

Daarnaast kan de kolom database_shard_id verder ook weggelaten worden. Deze kolom is een database artifact en geeft geen gevolg op het genre. Gelijkaardig met preview_url. Dit is enkel een weburl en geeft geen impact op het genre.

Van numerieke features kunnen meeste behouden worden doordat deze een grote impact hebben op genre.

In [ ]:
CURRENT_FEATURES = ['popularity', 'duration_ms', 'explicit', 'acousticness', 'danceability', 'energy', 
                    'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 
                    'tempo', 'time_signature', 'valence']
SAMPLE_GENRES = ['Classical', 'hard-rock', 'blues', 'country', 'latin','funk','disco','electronic','pop']

df_sample = df[df['genre'].isin(SAMPLE_GENRES)]

genre_means = df_sample.groupby('genre')[CURRENT_FEATURES].mean()
genre_means_z = (genre_means - genre_means.mean()) / genre_means.std()

### Feature Means by Genre

In de gegeven grafiek lijkt er een goeie spread over de verschillende genres over het gemiddelde van elke feature.
Enkele features die weinig verschil tonen is time_signature.
Samen met key en mode zijn dit geen ordinale waarde.
Key geeft de muzieknoot aan encoded in een nummer. Mode is dan of deze noot in major of in minor is.
Deze kunnen weliswaar invloed hebben op het genre maar moeten geencode worden omdat dit geen ordinale waarden zijn.

In [ ]:
plt.figure(figsize=(12, 8))
sn.heatmap(genre_means_z, cmap='coolwarm', center=0, annot=True, fmt='.1f')
plt.title('Feature means per genre (z-scored across genres)')
plt.tight_layout()
plt.savefig('./eda_visualisations/genre_feature_means.png', dpi=150)
plt.close()

display(Image(filename='./eda_visualisations/genre_feature_means.png'))

### Categorical Features Analysis

Key, mode, en time_signature zijn categorische features die geencodeerd moeten worden.

In [ ]:
CATEGORICAL_FEATURES = ['mode','key','time_signature']

plt.figure(figsize=(12, 4))
df_sample[CATEGORICAL_FEATURES].apply(lambda s: s.value_counts().sort_index()).plot(
    kind='bar', subplots=True, layout=(1,3), figsize=(12,4), legend=False)
plt.suptitle('Categorical Feature Distributions')
plt.tight_layout()
plt.savefig('./eda_visualisations/categorical_bars.png', dpi=150)
plt.close()

In [ ]:
display(Image(filename='./eda_visualisations/categorical_bars.png'))

Zoals in deze grafiek kan gezien worden zijn mode en key over het algemeen redelijk wijd verspreidt. time_signature aan de andere kant heeft de overgrote meerderheid op 4 met weinig variatie.
Doordat meeste tracks op time signature 4 bevinden zal het model wellicht geen goede voorspellingen kunnen maken op deze kolom. De waarden van mode en key kunnen wel behouden worden maar moeten geencode worden. Doordat er geen ordinaal verband is tussen verschillende muzieknoten

### Explicitness Analysis

Eer zijn maar een heel klein aantal nummers die explicit zijn over de volledige dataset geconcentreerd onder enkele genres.
Enkel kijken naar de samples dataframe geeft niks.
Doordat we al genres samen gaan groeperen zal deze feature zijn nut snel verliezen.

In [ ]:
plt.figure(figsize=(12, 5))
df.groupby('genre')['explicit'].value_counts().unstack().plot(kind='bar', stacked=True, figsize=(12,5))
plt.title('Explicit Tracks by Genre (Stacked)')
plt.tight_layout()
plt.savefig('./eda_visualisations/explicit.png', dpi=150)
plt.close()

display(Image(filename='./eda_visualisations/explicit.png'))

In [ ]:
explicit_by_genre = df.groupby('genre')['explicit'].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 5))
explicit_by_genre.head(15).plot(kind='bar', figsize=(12,5))
plt.ylabel('proportion explicit')
plt.title('Proportion of Explicit Tracks by Genre (Top 15)')
plt.tight_layout()
plt.savefig('./eda_visualisations/explicit_by_genre.png', dpi=150)
plt.close()

display(Image(filename='./eda_visualisations/explicit_by_genre.png'))

### Release Year Analysis

album_release_date:
De datum waarop een album is uitgebracht kan een impact hebben op het genre. Genres zoals jazz en funk zullen wellicht vaak meer oudere release dates hebben dan andere genres zoals rock.
Wellicht hebben we hier niet de volledige precisie voor nodig en volstaat het gewoon om het jaar op te geven.
Over het algemeen lijkt er wel een verband te zijn met wanneer de track uitgebracht werd en het genre.

In [ ]:
CURRENT_FEATURES = ['album_release_date', 'album_release_date_precision' ,'popularity', 'duration_ms', 
                    'genre', 'acousticness', 'danceability', 'energy', 'instrumentalness', 
                    'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence']

df_sample["release_year"] = pd.to_datetime(df_sample["album_release_date"], format="mixed", errors="coerce").dt.year

order = df_sample.groupby('genre')['release_year'].median().sort_values().index

plt.figure(figsize=(10, 8))
sn.boxplot(data=df_sample, y='genre', x='release_year', order=order)
plt.title('Release Year by Genre')
plt.tight_layout()
plt.savefig('./eda_visualisations/year_by_genre_box.png', dpi=150)
plt.close()

display(Image(filename='./eda_visualisations/year_by_genre_box.png'))

### Genre Grouping

Omdat de dataset enorm groot is en er per genre mogelijks maar enkele entries zijn is het best om deze te groeperen per genrefamilie.
De verdeling van genres is over het algemeen vrij subjectief en is er vaak overlap tussen verschillende groepen.
Over het algemeen worden genres gegroepeerd per 'familie' of in welk breder genre ze kunnen geplaatst worden.
De verschillende rock genres worden hier bv. onder Rock geplaatst.
Genres die geen overlappend genre hebben zoals Ambient werden hieruit weggelaten. Deze zouden te klein zijn in de dataset om er redelijke predicities mee te maken.
De verschillende genres zoals "French" en "German" werden hier ook uit weggelaten. Deze vertegenwoordigen eerder het land waar ze uit kwamen dan het effectieve genre.

In [ ]:
# Define genre mapping for grouping
GENRE_MAPPING = {
    # Rock genres
    "alt-rock": "Rock",
    "alternative": "Rock",
    "hard-rock": "Rock",
    "rock": "Rock",
    "punk": "Rock",
    "punk-rock": "Rock",
    "rock-n-roll": "Rock",
    "rockabilly": "Rock",
    "groove": "Rock",
    "grunge": "Rock",
    "goth": "Rock",
    "hardcore": "Rock",
    "psych-rock": "Rock",
    "power-pop": "Rock",
    "indie-rock": "Rock",
    "emo": "Rock",
    "j-rock": "Rock",
    "guitar": "Rock",
    
    # Metal genres
    "metal": "Metal",
    "metalcore": "Metal",
    "death-metal": "Metal",
    "black-metal": "Metal",
    "grindcore": "Metal",
    
    # Pop genres
    "pop": "Pop",
    "indie-pop": "Pop",
    "synth-pop": "Pop",
    "k-pop": "Pop",
    "j-pop": "Pop",
    "j-idol": "Pop",
    "cantopop": "Pop",
    "mandopop": "Pop",
    "pop-film": "Pop",
    
    # Electronic genres
    "dance": "Electronic",
    "edm": "Electronic",
    "electronic": "Electronic",
    "house": "Electronic",
    "deep-house": "Electronic",
    "techno": "Electronic",
    "trance": "Electronic",
    "dubstep": "Electronic",
    "drum-and-bass": "Electronic",
    "breakbeat": "Electronic",
    "chicago-house": "Electronic",
    "detroit-techno": "Electronic",
    "minimal-techno": "Electronic",
    "progressive-house": "Electronic",
    "club": "Electronic",
    "dub": "Electronic",
    "idm": "Electronic",
    "hardstyle": "Electronic",
    
    # Hip-Hop/R&B genres
    "hip-hop": "Hip-Hop/R&B",
    "r-n-b": "Hip-Hop/R&B",
    "reggae": "Hip-Hop/R&B",
    "reggaeton": "Hip-Hop/R&B",
    "dancehall": "Hip-Hop/R&B",
    
    # Jazz/Blues genres
    "jazz": "Jazz/Blues",
    "blues": "Jazz/Blues",
    
    # Classical/Opera genres
    "classical": "Classical/Opera",
    "opera": "Classical/Opera",
    "piano": "Classical/Opera",
    "romance": "Classical/Opera",
    "show-tunes": "Classical/Opera",
    
    # Country/Folk genres
    "country": "Country/Folk",
    "folk": "Country/Folk",
    "bluegrass": "Country/Folk",
    "honky-tonk": "Country/Folk",
    "singer-songwriter": "Country/Folk",
    "songwriter": "Country/Folk",
    
    # Latin genres
    "latin": "Latin",
    "latino": "Latin",
    "salsa": "Latin",
    "samba": "Latin",
    "brazil": "Latin",
    "forro": "Latin",
    "mpb": "Latin",
    "pagode": "Latin",
    "sertanejo": "Latin",
    
    # Funk/Disco genres
    "funk": "Funk/Disco",
    "disco": "Funk/Disco",
}

# 2. Framework Setup

This section defines the common framework used by all models, including data preprocessing, training, evaluation, and visualization utilities.

In [ ]:
# Define constants
DATA_PATH = 'Spotify_tracks_final.csv'
TARGET = 'genre'
AUDIO_FEATURES = [
    "loudness", "acousticness", "energy", "instrumentalness",
    "danceability", "valence", "release_year", "duration_ms",
    "tempo", "liveness", "speechiness", "popularity", "mode", "key",
]

In [ ]:
def plot_confusion_matrix(y_true, y_pred, labels, out_path):
    """
    Plot and save confusion matrix.
    
    Note: This function saves the confusion matrix to the confusion_matrices directory.
    In this notebook, we'll display placeholders since the confusion matrices are not yet complete.
    """
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(10, 10))
    ConfusionMatrixDisplay(cm, display_labels=labels).plot(
        ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False
    )
    fig.savefig(f"./confusion_matrices/{out_path}.png")
    plt.close(fig)
    return fig

In [ ]:
def evaluate(model, X_test, y_test, out_path):
    """
    Evaluate a trained model on test data.
    Prints accuracy, classification report, and permutation importances.
    Also generates and saves confusion matrix.
    """
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy:.3f}")
    print(classification_report(y_test, y_pred, zero_division=0))
    
    # Calculate permutation importances
    result = permutation_importance(
        model, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1
    )
    importances = pd.Series(result.importances_mean, index=X_test.columns)
    print("\nPermutation importances (test set):")
    print(importances.sort_values(ascending=False))
    
    # Generate confusion matrix
    plot_confusion_matrix(y_test, y_pred, model.classes_, out_path)
    
    return accuracy, importances

In [ ]:
def train_model(model, out_path, param_field):
    """
    Train a model with hyperparameter tuning using RandomizedSearchCV.
    
    This function:
    1. Loads and preprocesses the data
    2. Sets up preprocessing pipeline
    3. Performs randomized search with cross-validation
    4. Evaluates on test set
    5. Saves confusion matrix
    """
    # Load and preprocess data
    df = (
        pd.read_csv(DATA_PATH, index_col=0)
        .drop_duplicates(subset=["track_name"], keep="first")
        .assign(**{TARGET: lambda d: d[TARGET].map(GENRE_MAPPING)})
        .dropna(subset=[TARGET])
        .copy()
    )

    # Add release year from album release date
    df["release_year"] = pd.to_datetime(
        df["album_release_date"], format="mixed", errors="coerce"
    ).dt.year

    y = df[TARGET]
    X = df[AUDIO_FEATURES]

    # Encoding van key. De waarde van mode moet niet encode worden want dit is al een binaire waarde.
    preprocess = ColumnTransformer(
        [("key_ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["key"])],
        remainder="passthrough",
        verbose_feature_names_out=False,
    )

    # Split van 70 - 15 - 15 (train - val - test)
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X,
        y,
        test_size=0.15,
        stratify=y,
        random_state=42
    )
    # 0.15 / 0.85 = 0.17647...
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val,
        y_train_val,
        test_size=0.176,
        stratify=y_train_val,
        random_state=42
    )
    print(f"Dataset: {len(X)} samples ({len(X_train)} train, {len(X_test)} test, {len(X_val)} val)\n")

    # Pipeline voor het toepasssen van de encoding. Doet eerst encoding voor het model gegeven wordt.
    pipeline = Pipeline([
        ("preprocess", preprocess),
        ("model", model),
    ])

    # Randomized tuning. Grid tuning over de volledige dataset zou te lang duren.
    # Cross validation over 5 folds. Testset opgedeeld in 5 folds en telkens getest met gegeven parameters.
    # Wordt cross-validated met andere folds.
    gs = RandomizedSearchCV(
        estimator=pipeline, 
        cv=5, 
        param_distributions=param_field, 
        scoring="accuracy", 
        n_jobs=-1, 
        random_state=42, 
        refit=True
    )
    
    # Kleinere dataset voor sneller klaar te zijn (ik heb maar 8 cores :( )
    idx = X_train.sample(n=100000, random_state=42).index
    gs.fit(X_train.loc[idx], y_train.loc[idx])
    print(gs.best_params_, gs.best_score_)
    
    best_model = gs.best_estimator_
    validation_accuracy = best_model.score(X_val, y_val)
    print(f"Validation accuracy: {validation_accuracy}")

    # Train on full training set and evaluate
    best_model.fit(X_train, y_train)
    return best_model, X_test, y_test, out_path

# 3. Model Definitions

This section defines the three machine learning models used for genre classification.

## 3.1 HistGradientBoosting Classifier

Waarom histgradientboosting?
Decision trees zijn een goeie manier om de gegeven features te classifiseren. Een genre hangt sterk af van verschillende features.
Een genre bv met hoge energie, lage dansbaarheid en een recent release jaar kan eenvoudigt gemapt worden aan een decision tree.
Het model bouwt onderliggende de verschillende "beslissingen" die leiden naar een specifiek genre op. Bij voorspellingen maken kan dan door de flowchart gegaan worden tot het genre bereikt wordt.
Bij histgradientboosting worden telkens bomen opgebouwd die op elkaar itereren. Nieuwe decision trees leren telkens van de fouten uit vorige decision trees.
Op het einde hebben we dus vervolgens een decision tree dat de som is van alle voorgaande decision trees.

In [ ]:
# Define parameter grid for HistGradientBoosting
HISTGRADIENT_PARAMS = {
    "model__learning_rate": loguniform(1e-2, 3e-1),
    "model__max_iter": randint(200, 800),
    "model__max_leaf_nodes": randint(15, 127),
    "model__max_depth": [None, 6, 10, 14],
    "model__min_samples_leaf": randint(5, 100),
    "model__l2_regularization": loguniform(1e-3, 10),
    "model__max_bins": [127, 255],
}

def histgradient():
    model = HistGradientBoostingClassifier(
        min_samples_leaf=5,
        random_state=42
    )
    return model, HISTGRADIENT_PARAMS, "confusion_matrix_histgradientboost"

## 3.2 K-Nearest Neighbors Classifier

Bij KNN plotten we elk punt op een grafiek en kijken hoeveel x aantal van de dichtste buren gelijk zijn aan een specifiek genre.
Gelijkaardige genres zouden dicht bij elkaar moeten liggen op deze grafiek doordat ze gelijkaardige features hebben.
Enkele problemen hiermee zouden mogelijks kunnen zijn is overlap tussen verschillende genres. Door genres samen te groeperen zullen mogelijks genres die aan de rand staan van een bepaalde groep mogelijks mislabeled worden.

In [ ]:
# Define parameter grid for KNN
KNN_PARAMS = {
    "model__n_neighbors": randint(5, 100),
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2],                      # Manhattan vs Euclidean
    "model__algorithm": ["auto", "ball_tree", "kd_tree"],
    "model__leaf_size": randint(20, 60),
}

def knn():
    model = KNeighborsClassifier(n_neighbors=30, n_jobs=-1)
    return model, KNN_PARAMS, "confusion_matrix_knn"

## 3.3 Multi-Layer Perceptron (Neural Network)

Neuraal netwerk. Neemt de verschillende audio features als input.
Door onderliggende de weights tussen de neurons aan te passen en gewichten toe te kennen kunnen de verschillende audio features door het netwerk gaan en vervolgens een voorspelling te doen.
Gelijkaardig aan decision trees kan het model zo verbanden op te bouwen tussen verschillende features en hun genre.

In [ ]:
# Define parameter grid for MLP
MLP_PARAMS = {
    "model__hidden_layer_sizes": [(64, 32), (128, 64), (128, 64, 32), (256, 128), (64,)],
    "model__activation": ["relu", "tanh"],
    "model__alpha": loguniform(1e-5, 1e-1),
    "model__learning_rate_init": loguniform(1e-4, 1e-2),
    "model__batch_size": [128, 256, 512],
    "model__n_iter_no_change": [10, 20],
}

def mlp():
    model = MLPClassifier(
        hidden_layer_sizes=(64, 32), 
        max_iter=1000,
        early_stopping=True, 
        random_state=42
    )
    return model, MLP_PARAMS, "confusion_matrix_mlp"

# 4. Model Training and Evaluation

Now we'll train all three models and evaluate their performance.

In [ ]:
# Initialize models
print("=====Initializing Models=====")

# Get model definitions
hist_model, hist_params, hist_out = histgradient()
knn_model, knn_params, knn_out = knn()
mlp_model, mlp_params, mlp_out = mlp()

print("Models initialized successfully")

In [ ]:
# Note: Training all models may take a long time depending on your hardware.
# The following cells train each model separately.
# You can run them individually to save time.

# For demonstration purposes, we'll show the structure but note that
# actual training requires significant computational resources.

## 4.1 Train HistGradientBoosting Model

In [ ]:
print("=====HistGradientBoosting=====\n")

# Train the model
best_hist_model, X_test_hist, y_test_hist, _ = train_model(hist_model, hist_out, hist_params)

# Evaluate
hist_accuracy, hist_importances = evaluate(best_hist_model, X_test_hist, y_test_hist, hist_out)

## 4.2 Train KNN Model

In [ ]:
print("=====KNN=====\n")

# Train the model
best_knn_model, X_test_knn, y_test_knn, _ = train_model(knn_model, knn_out, knn_params)

# Evaluate
knn_accuracy, knn_importances = evaluate(best_knn_model, X_test_knn, y_test_knn, knn_out)

## 4.3 Train MLP Model

In [ ]:
print("=====MLP=====\n")

# Train the model
best_mlp_model, X_test_mlp, y_test_mlp, _ = train_model(mlp_model, mlp_out, mlp_params)

# Evaluate
mlp_accuracy, mlp_importances = evaluate(best_mlp_model, X_test_mlp, y_test_mlp, mlp_out)

# 5. Results and Visualizations

This section displays the results from the EDA and model training.

## 5.1 EDA Visualizations

Below are all the visualizations generated during the EDA phase.

In [ ]:
# Display all EDA visualizations
eda_images = [
    './eda_visualisations/genre_similarity_heatmap.png',
    './eda_visualisations/heatmap_duration.png',
    './eda_visualisations/genre_feature_means.png',
    './eda_visualisations/categorical_bars.png',
    './eda_visualisations/explicit.png',
    './eda_visualisations/explicit_by_genre.png',
    './eda_visualisations/year_by_genre_box.png'
]

print("EDA Visualizations:\n")
for img in eda_images:
    print(f"- {img}")

In [ ]:
# Display each visualization
for img_path in eda_images:
    try:
        display(Image(filename=img_path))
        print(f"\n**{img_path.split('/')[-1]}**\n")
    except Exception as e:
        print(f"Could not display {img_path}: {e}")

## 5.2 Confusion MatricesBelow are the confusion matrices generated from model training and evaluation.

In [ ]:
# Display HistGradientBoosting Confusion Matrixprint("### HistGradientBoosting Confusion Matrix\n")display(Image(filename="./confusion_matrices/confusion_matrix_histgradientboost.png"))

In [ ]:
# Display KNN Confusion Matrixprint("### KNN Confusion Matrix\n")display(Image(filename="./confusion_matrices/confusion_matrix_knn.png"))

In [ ]:
# Display MLP Confusion Matrixprint("### MLP Confusion Matrix\n")display(Image(filename="./confusion_matrices/confusion_matrix_mlp.png"))

## 5.3 Model Comparison

After running all model training cells, you can compare the results here.

In [ ]:
# Model comparison summary (will be populated after running all models)
print("### Model Performance Summary\n")
print("Note: Run all model training cells to see actual results.\n")

# Placeholder comparison
results_summary = """
| Model | Validation Accuracy | Test Accuracy | Best Parameters |
|-------|---------------------|---------------|-----------------|
| HistGradientBoosting | [Will appear after training] | [Will appear after training] | [Will appear after training] |
| KNN | [Will appear after training] | [Will appear after training] | [Will appear after training] |
| MLP | [Will appear after training] | [Will appear after training] | [Will appear after training] |
"""

print(results_summary)

## 5.4 Feature Importance

Permutation importances will be displayed after model training.

In [ ]:
# Display feature importance from permutation importance
print("Feature importance analysis will appear here after training each model.\n")
print("Each model's evaluate() function prints the permutation importances for the test set.")

# 6. Main Execution

This section provides a convenient way to run all models sequentially, similar to the main.py script.

In [ ]:
def main():
    """
    Run all models sequentially and collect results.
    This is equivalent to running main.py.
    """
    results = {}
    
    print("=====Histgradient=====\n")
    model, params, out = histgradient()
    best_model, X_test, y_test, _ = train_model(model, out, params)
    acc, imp = evaluate(best_model, X_test, y_test, out)
    results['HistGradientBoosting'] = {'accuracy': acc, 'importances': imp}
    
    print("=====KNN=====\n")
    model, params, out = knn()
    best_model, X_test, y_test, _ = train_model(model, out, params)
    acc, imp = evaluate(best_model, X_test, y_test, out)
    results['KNN'] = {'accuracy': acc, 'importances': imp}
    
    print("=====MLP=====\n")
    model, params, out = mlp()
    best_model, X_test, y_test, _ = train_model(model, out, params)
    acc, imp = evaluate(best_model, X_test, y_test, out)
    results['MLP'] = {'accuracy': acc, 'importances': imp}
    
    return results

# Note: Running this cell will train all models, which may take significant time.
# Uncomment the line below to run all models:
# results = main()

# Usage Instructions

## How to Use This Notebook

1. **Run EDA Cells**: Execute all cells in Section 1 to perform exploratory data analysis and generate visualizations.
   - All EDA visualizations will be saved to the `eda_visualisations/` directory
   - The visualizations will also be displayed inline in the notebook

2. **Run Framework Cells**: Execute cells in Section 2 to set up the common framework.
   - This defines helper functions for training and evaluation

3. **Run Model Definition Cells**: Execute cells in Section 3 to define the three models.

4. **Train Models Individually**: Execute the training cells in Section 4 to train each model.
   - Each model training cell may take several minutes depending on your hardware
   - Confusion matrices will be saved to the `confusion_matrices/` directory

5. **View Results**: Check Section 5 for all visualizations and results.
   - EDA visualizations are displayed immediately
   - Confusion matrix placeholders will be replaced with actual images after training

## Notes

- The notebook is designed to work with the provided dataset (`Spotify_tracks_final.csv`)
- All code from the repository has been adapted to work in a notebook environment
- Comments from the original Python files have been converted to markdown cells
- The confusion matrices currently show placeholders and will be generated during training